# Pose spike: rtmlib vs Keypoint R-CNN (T1.5)

Picks the pose backend with numbers (docs/phases/phase-1-foundations.md, T1.5). Run it on
the RTX 4050 laptop and on a Colab T4, on both consented test rallies, then copy the table
into TECH_STACK.md ("Pose spike results") and set `stages.pose.backend` / `mode` in
`configs/pipeline.yaml`.

Both backends go through `shuttlesense.pose.estimator.load_detector`, the same loader the
pipeline uses, so what is measured here is what ships. **Clear all outputs before
committing** (the frames are footage of people).

Acceptance: both players detected in at least 90% of frames on both clips, and at least
10 fps on the T4.

In [ ]:
from pathlib import Path
import time

import numpy as np

from shuttlesense.io import FrameReader
from shuttlesense.pose.estimator import load_detector
from shuttlesense.pose.tracking import filter_players

CLIPS = [Path("../../samples/rally01.mp4"), Path("../../samples/rally02.mp4")]
DEVICE = "cuda"  # "cpu" works, but the fps would not mean anything
MAX_FRAMES = 900  # 30 s at 30 fps per clip is enough for the rates
CANDIDATES = [("rtmlib", "balanced"), ("rtmlib", "performance"), ("keypoint_rcnn", "balanced")]
MIN_PERSON_SCORE = 0.4  # same as configs/pipeline.yaml stages.pose.min_person_score

## Run every candidate on every clip

For each frame: people above the score threshold, kept by the same court filter as the
pipeline (no calibration here, so the lower-90%-of-frame rule), and the time per frame.
The first frames warm the model up and are not timed.

In [ ]:
WARMUP = 10


def measure(backend, mode, clip):
    detect = load_detector(backend, mode, DEVICE)
    reader = FrameReader(clip)
    times, both, kept_frames = [], 0, []
    for i, frame in reader.frames(0, min(len(reader), MAX_FRAMES)):
        start = time.perf_counter()
        people = [p for p in detect(frame) if p.score >= MIN_PERSON_SCORE]
        elapsed = time.perf_counter() - start
        players = filter_players(people, frame_height=frame.shape[0])
        if i >= WARMUP:
            times.append(elapsed)
        both += len(players) == 2
        kept_frames.append((i, players))
    n = len(kept_frames)
    return {
        "backend": backend, "mode": mode, "clip": clip.name, "frames": n,
        "fps": round(1 / np.mean(times), 1) if times else None,
        "both_players_pct": round(100 * both / n, 1) if n else None,
        "kept": kept_frames,
    }


results = []
for backend, mode in CANDIDATES:
    for clip in CLIPS:
        try:
            results.append(measure(backend, mode, clip))
        except Exception as exc:  # a backend that is not installed is reported, not fatal
            print(f"{backend}/{mode} on {clip.name}: {exc}")

## Results table (paste into TECH_STACK.md)

In [ ]:
gpu = "unknown"
try:
    import torch

    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
except ImportError:
    pass
print(f"GPU: {gpu}\n")
print("| Backend | Mode | Clip | Frames | fps | Both players detected |")
print("| --- | --- | --- | --- | --- | --- |")
for r in results:
    print(f"| {r['backend']} | {r['mode']} | {r['clip']} | {r['frames']} | {r['fps']} | "
          f"{r['both_players_pct']}% |")

## Visual check: 30 random frames per candidate

The far player is the hard case; also look at frames during smashes (motion blur). Boxes
are drawn on the frames; nothing is saved to disk.

In [ ]:
import cv2
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)


def show(result, count=30):
    reader = FrameReader(Path("../../samples") / result["clip"])
    picks = sorted(rng.choice(len(result["kept"]), size=min(count, len(result["kept"])),
                              replace=False))
    cols = 6
    fig, axes = plt.subplots((len(picks) + cols - 1) // cols, cols, figsize=(18, 10))
    for ax, k in zip(axes.flat, picks, strict=False):
        i, players = result["kept"][k]
        frame = reader.batch(i, 1)[0].copy()
        for p in players:
            x, y, w, h = map(int, p.bbox)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        ax.imshow(frame[:, :, ::-1])
        ax.set_title(f"{i}: {len(players)} found", fontsize=8)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(f"{result['backend']} / {result['mode']} on {result['clip']}")
    plt.show()


for r in results:
    show(r)

## Decision

Write it in TECH_STACK.md with the table above and update decision D6 in DECISIONS.md:
which backend and mode, its numbers on both GPUs, and what was seen for the far player.
Then set `stages.pose.backend` and `mode` in `configs/pipeline.yaml`.